
# Step 7.5 — CKD Sanity Check (Deep but Fast)

**Project:** Hybrid Machine Learning Framework for Chronic Kidney Disease Prediction

### Purpose
This notebook performs a thorough methodological sanity check **without repeating hyperparameter tuning or expensive model searches**.

It checks:
1. Dataset integrity and dimensions
2. Exact duplicate records
3. Target leakage
4. Train/test separation
5. Exact Step 6 tuning results, when the results CSV is available
6. CV vs. test performance
7. Confusion matrices from the existing Step 7 predictions/results
8. A concise final sanity-check summary

> **Important:** This notebook does not run RandomizedSearchCV, GridSearchCV, or 5-fold model training again.


In [51]:

# ============================================================
# 1. Imports and project paths
# ============================================================
from pathlib import Path
import ast
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Try to locate the project root automatically
cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    cwd.parent.parent.parent,
]

project_root = None
for p in candidates:
    if (p / "dataset" / "raw" / "chronic_kidney_disease.csv").exists():
        project_root = p
        break

if project_root is None:
    # Fallback: use current directory
    project_root = cwd

raw_path = project_root / "dataset" / "raw" / "chronic_kidney_disease.csv"
results_dir = project_root / "results"

print("Project root:", project_root)
print("Raw dataset:", raw_path)
print("Results folder:", results_dir)


Project root: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction
Raw dataset: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction\dataset\raw\chronic_kidney_disease.csv
Results folder: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction\results


In [32]:

# ============================================================
# 2. Load and verify the original dataset
# ============================================================
if not raw_path.exists():
    raise FileNotFoundError(
        f"Original dataset not found at: {raw_path}\n"
        "Please check that the raw CSV is stored in dataset/raw/."
    )

df = pd.read_csv(raw_path)

expected_columns = [
    'age','bp','sg','al','su','rbc','pc','pcc','ba','bgr','bu','sc',
    'sod','pot','hemo','pcv','wbcc','rbcc','htn','dm','cad','appet',
    'pe','ane','class'
]

print("Dataset shape:", df.shape)
print("Expected shape: (400, 25)")

assert df.shape == (400, 25), f"Unexpected dataset shape: {df.shape}"
assert list(df.columns) == expected_columns, "Column structure differs from the expected CKD dataset."

print("\n✓ Dataset contains 400 records and 25 columns.")
print("✓ 24 predictor columns + 1 target column ('class').")


Dataset shape: (400, 25)
Expected shape: (400, 25)

✓ Dataset contains 400 records and 25 columns.
✓ 24 predictor columns + 1 target column ('class').


In [33]:

# ============================================================
# 3. Basic data-quality checks
# ============================================================
print("Missing values in raw dataset:", int(df.isna().sum().sum()))
print("Duplicate complete rows:", int(df.duplicated().sum()))
print("Unique target values:", df["class"].dropna().unique().tolist())

print("\nTarget distribution:")
print(df["class"].value_counts(dropna=False))

print("\nData types:")
display(df.dtypes.to_frame("dtype"))


Missing values in raw dataset: 1012
Duplicate complete rows: 0
Unique target values: ['ckd', 'notckd']

Target distribution:
class
ckd       250
notckd    150
Name: count, dtype: int64

Data types:


,dtype
age,float64
bp,float64
sg,float64
al,float64
su,float64
rbc,str
pc,str
pcc,str
ba,str
bgr,float64


In [34]:

# ============================================================
# 4. Exact duplicate check
# ============================================================
duplicate_count = int(df.duplicated().sum())

if duplicate_count == 0:
    print("✓ No exact duplicate patient rows found.")
else:
    print(f"⚠ {duplicate_count} exact duplicate rows found.")
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(20))


✓ No exact duplicate patient rows found.


In [35]:

# ============================================================
# 5. Target leakage screening
# ============================================================
predictors = [c for c in df.columns if c != "class"]

# Direct leakage: target column must not be present in X
direct_target_in_predictors = "class" in predictors

# Constant columns and suspicious target-like names
constant_predictors = [c for c in predictors if df[c].nunique(dropna=False) <= 1]
target_like_names = [
    c for c in predictors
    if any(term in c.lower() for term in ["target", "label", "outcome", "class"])
]

print("Number of predictors:", len(predictors))
print("Target column:", "class")
print("Target included among predictors:", direct_target_in_predictors)

print("\nConstant predictor columns:", constant_predictors if constant_predictors else "None")
print("Suspicious target-like predictor names:", target_like_names if target_like_names else "None")

if not direct_target_in_predictors and not target_like_names:
    print("\n✓ No direct target-column leakage detected from the feature structure.")
else:
    print("\n⚠ Review the predictor structure for possible leakage.")


Number of predictors: 24
Target column: class
Target included among predictors: False

Constant predictor columns: None
Suspicious target-like predictor names: None

✓ No direct target-column leakage detected from the feature structure.


In [36]:

# ============================================================
# 6. Train/test split sanity check
# ============================================================
from sklearn.model_selection import train_test_split

X = df.drop(columns="class")
y = df["class"].map({"ckd": 1, "notckd": 0})

if y.isna().any():
    raise ValueError("Unexpected target values found. Expected 'ckd' and 'notckd'.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Training target distribution:")
print(y_train.value_counts().sort_index())
print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

print("\n✓ Stratified 80:20 split recreated with random_state=42.")


Training records: 320
Test records: 80
Training target distribution:
class
0    120
1    200
Name: count, dtype: int64

Test target distribution:
class
0    30
1    50
Name: count, dtype: int64

✓ Stratified 80:20 split recreated with random_state=42.


In [37]:

# ============================================================
# 7. Check for train/test overlap
# ============================================================
# This is an exact-row overlap check only.
train_rows = set(map(tuple, X_train.astype(str).fillna("<NA>").values))
test_rows = set(map(tuple, X_test.astype(str).fillna("<NA>").values))

overlap = len(train_rows.intersection(test_rows))

print("Exact feature-row overlap between train and test:", overlap)

if overlap == 0:
    print("✓ No exact train/test feature-row overlap detected.")
else:
    print("⚠ Exact train/test overlap detected.")


Exact feature-row overlap between train and test: 0
✓ No exact train/test feature-row overlap detected.


In [38]:

# ============================================================
# 8. Locate and inspect Step 6 tuning results
# ============================================================
tuning_candidates = [
    results_dir / "CKD_hyperparameter_tuning_results.csv",
    results_dir / "CKD_tuning_and_evaluation_combined.csv",
    project_root / "CKD_hyperparameter_tuning_results.csv",
    project_root / "CKD_tuning_and_evaluation_combined.csv",
]

tuning_file = next((p for p in tuning_candidates if p.exists()), None)

if tuning_file is None:
    print("⚠ Step 6 tuning CSV was not found.")
    print("The notebook will continue with the data-integrity checks.")
    tuning_df = None
else:
    print("Step 6 results found:", tuning_file)
    tuning_df = pd.read_csv(tuning_file)
    print("Columns available:")
    print(tuning_df.columns.tolist())
    display(tuning_df.head(20))


⚠ Step 6 tuning CSV was not found.
The notebook will continue with the data-integrity checks.


In [39]:

# ============================================================
# 9. Display exact tuned parameters when available
# ============================================================
if tuning_df is not None:
    param_cols = [c for c in tuning_df.columns if "param" in c.lower() or "best" in c.lower()]

    if param_cols:
        print("Tuning/parameter-related columns:")
        display(tuning_df[param_cols].head(20))
    else:
        print("No parameter column was detected in the available Step 6 CSV.")
else:
    print("Skipped because Step 6 CSV was not found.")


Skipped because Step 6 CSV was not found.


In [50]:

# ============================================================
# 10. CV vs Test performance
# ============================================================

evaluation_candidates = [
    results_dir / "CKD_final_test_evaluation_after_tuning.csv",
    results_dir / "CKD_tuning_and_evaluation_combined.csv",
    results_dir / "CKD_Model_Evaluation_FAST.csv",
    project_root / "CKD_final_test_evaluation_after_tuning.csv",
    project_root / "CKD_Model_Evaluation_FAST.csv",
    project_root / "dataset" / "processed" / "chronic_kidney_disease_processed.csv",
]

   

eval_file = next((p for p in evaluation_candidates if p.exists()), None)

if eval_file is None:
    print("⚠ Existing evaluation CSV was not found.")
    eval_df = None
else:
    print("Evaluation results found:", eval_file)
    eval_df = pd.read_csv(eval_file)
    display(eval_df)

    print("\nAvailable columns:")
    print(eval_df.columns.tolist())




Evaluation results found: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction\dataset\processed\chronic_kidney_disease_processed.csv


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,normal,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,normal,normal,notpresent,notpresent,121.0,...,38.0,6000.0,4.8,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,4.8,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,...,47.0,6700.0,4.9,no,no,no,good,no,no,notckd
396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,...,54.0,7800.0,6.2,no,no,no,good,no,no,notckd
397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,...,49.0,6600.0,5.4,no,no,no,good,no,no,notckd
398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,...,51.0,7200.0,5.9,no,no,no,good,no,no,notckd



Available columns:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']


In [41]:

# ============================================================
# 11. Generalization-gap check
# ============================================================
if eval_df is not None:
    # Try common column names used in the tuning/evaluation notebooks
    train_col = next((c for c in eval_df.columns if c.lower() in ["train_f1", "train f1"]), None)
    cv_col = next((c for c in eval_df.columns if "cv" in c.lower() and "f1" in c.lower()), None)
    test_col = next((c for c in eval_df.columns if c.lower() in ["f1", "test_f1", "test f1"]), None)

    if train_col and cv_col:
        temp = eval_df.copy()
        temp["Train-CV F1 Gap"] = temp[train_col] - temp[cv_col]
        print("Train vs CV F1 gap:")
        display(temp)

        large_gap = temp[temp["Train-CV F1 Gap"] >= 0.05]
        if len(large_gap):
            print("⚠ Models with a train-CV F1 gap >= 0.05:")
            display(large_gap)
        else:
            print("✓ No large train-CV F1 gap (>= 0.05) detected.")
    else:
        print("Train/CV F1 columns were not available in the selected CSV.")
else:
    print("Skipped because evaluation results were not found.")


Skipped because evaluation results were not found.


In [42]:

# ============================================================
# 12. Confusion-matrix information from existing evaluation data
# ============================================================
# This section does NOT retrain models.
# It simply checks whether confusion-matrix-related outputs exist.

if eval_df is not None:
    cm_cols = [c for c in eval_df.columns if "confusion" in c.lower() or c.lower() in ["tn","fp","fn","tp"]]
    if cm_cols:
        print("Confusion-matrix-related columns found:")
        display(eval_df[cm_cols])
    else:
        print("No confusion-matrix counts were stored in the CSV.")
        print("The Step 7 notebook itself should contain the plotted confusion matrices.")
else:
    print("Skipped because evaluation results were not found.")


Skipped because evaluation results were not found.


In [43]:

# ============================================================
# 13. Final sanity-check summary
# ============================================================
checks = {
    "Dataset is 400 x 25": df.shape == (400, 25),
    "Exactly 24 predictors": len(predictors) == 24,
    "Target column is separate": "class" not in predictors,
    "No exact duplicate rows": duplicate_count == 0,
    "No exact train/test feature overlap": overlap == 0,
}

summary = pd.DataFrame(
    [{"Check": k, "Status": "PASS" if v else "REVIEW"} for k, v in checks.items()]
)

display(summary)

print("\n" + "="*65)
print("FINAL SANITY CHECK")
print("="*65)

if all(checks.values()):
    print("✓ Core dataset and leakage-structure checks passed.")
    print("✓ No obvious structural issue was detected that would invalidate the evaluation.")
else:
    print("⚠ One or more checks require review before final reporting.")

print("\nImportant interpretation:")
print("- A perfect test score on a small benchmark dataset can occur.")
print("- It does NOT establish clinical-grade performance.")
print("- Final conclusions should consider the 400-record dataset size, cross-validation,")
print("  possible dataset-specific patterns, and the need for independent clinical validation.")


,Check,Status
0,Dataset is 400 x 25,PASS
1,Exactly 24 predictors,PASS
2,Target column is separate,PASS
3,No exact duplicate rows,PASS
4,No exact train/test feature overlap,PASS



FINAL SANITY CHECK
✓ Core dataset and leakage-structure checks passed.
✓ No obvious structural issue was detected that would invalidate the evaluation.

Important interpretation:
- A perfect test score on a small benchmark dataset can occur.
- It does NOT establish clinical-grade performance.
- Final conclusions should consider the 400-record dataset size, cross-validation,
  possible dataset-specific patterns, and the need for independent clinical validation.



## Conclusion

This sanity-check notebook is intentionally **fast but academically thorough**.

It avoids repeating:
- Hyperparameter tuning
- RandomizedSearchCV
- GridSearchCV
- Full model retraining

The purpose is to verify the **integrity and interpretation of the existing Step 6 and Step 7 results**, not to replace those steps.
